In [ ]:
#Preprocessing

In [1]:
#Import needed libraries
import os
import cv2
import numpy as np
from tensorflow.keras.utils import to_categorical

C:\Users\Siraj\anaconda3\Lib\site-packages\keras\src\export\tf2onnx_lib.py:8: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, "object"):


In [1]:
#Load the dataset
data_path=r"C:\Users\Siraj\Downloads\data\data


In [3]:
categories=os.listdir(data_path)

In [4]:
labels=[i for i in range(len(categories))]


In [5]:
label_dict=dict(zip(categories,labels))


In [6]:
print(categories)

['without_mask', 'with_mask']


In [7]:
print(labels)

[0, 1]


In [8]:
print(label_dict)

{'without_mask': 0, 'with_mask': 1}


In [9]:
image_size=100
data=[]
target=[]

for category in categories:
        folder_path=os.path.join(data_path,category)
        image_names=os.listdir(folder_path)
        for image_name in image_names:
            image_path=os.path.join(folder_path,image_name)
            img=cv2.imread(image_path)
            try:
                grey=cv2.cvtColor(img,cv2.COLOR_BGR2GRAY)
                resize=cv2.resize(grey,(image_size,image_size))
                data.append(resize)
                target.append(label_dict[category])
            except:
                print("Exception",e)

In [10]:
data=np.array(data)/255.0
data=np.reshape(data,(data.shape[0],image_size,image_size,1))
target=np.array(target)

In [11]:
new_target=to_categorical(target)

In [12]:
np.save('data.npy', data)

In [13]:
np.save('target.npy', target)  

In [14]:
#Training

In [15]:
import numpy as np
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense,Activation,Flatten,Dropout
from tensorflow.keras.layers import Conv2D,MaxPooling2D
from tensorflow.keras.callbacks import ModelCheckpoint
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.models import load_model
data=np.load('data.npy')
target=np.load('target.npy')

In [16]:
model=Sequential()

#create a convolutional layer
model.add(Conv2D(200,(3,3),input_shape=data.shape[1:]))#Here 200 convolutional layer
model.add(Activation('relu'))#relu is the activation function
model.add(MaxPooling2D(pool_size=(2,2)))#2x2 maxpooling layer
#The first CNN layer followed by Relu and MaxPooling layers

model.add(Conv2D(100,(3,3)))
model.add(Activation('relu'))
model.add(MaxPooling2D(pool_size=(2,2)))
#The second convolution layer followed by Relu and MaxPooling layers

model.add(Flatten())
model.add(Dropout(0.5))
#Flatten layer to stack the output convolutions from second convolution layer
model.add(Dense(50,activation='relu'))
#Dense layer of 50 neurons
model.add(Dense(2,activation='softmax'))
#The Final layer with two outputs for two categories

model.compile(loss='sparse_categorical_crossentropy',optimizer='adam',metrics=['accuracy'])


C:\Users\Siraj\anaconda3\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [17]:
from sklearn.model_selection import train_test_split
#data preperation
train_data,test_data,train_target,test_target=train_test_split(data,target,test_size=0.1)

In [ ]:
checkpoint = ModelCheckpoint('model-{epoch:03d}.keras',monitor='val_loss',verbose=0,save_best_only=True,mode='auto')
history=model.fit(train_data,train_target,epochs=5,callbacks=[checkpoint],validation_split=0.2)

Epoch 1/5
170/170 ━━━━━━━━━━━━━━━━━━━━ 101s 571ms/step - accuracy: 0.5968 - loss: 0.6560 - val_accuracy: 0.6787 - val_loss: 0.5913
Epoch 2/5
170/170 ━━━━━━━━━━━━━━━━━━━━ 95s 560ms/step - accuracy: 0.6860 - loss: 0.5781 - val_accuracy: 0.7544 - val_loss: 0.5141
Epoch 3/5
170/170 ━━━━━━━━━━━━━━━━━━━━ 96s 566ms/step - accuracy: 0.7550 - loss: 0.4972 - val_accuracy: 0.8162 - val_loss: 0.4230
Epoch 4/5
170/170 ━━━━━━━━━━━━━━━━━━━━ 94s 553ms/step - accuracy: 0.8225 - loss: 0.3921 - val_accuracy: 0.8301 - val_loss: 0.3711
Epoch 5/5
 83/170 ━━━━━━━━━━━━━━━━━━━━ 16:26 11s/step - accuracy: 0.8567 - loss: 0.3266

In [18]:
from matplotlib import pyplot as plt
plt.plot(history.history['loss'],'r',label='training loss')
plt.plot(history.history['val_loss'],label='validation loss')
plt.xlabel('# epochs')
plt.ylabel('loss')
plt.legend()
plt.show()

NameError: name 'history' is not defined

In [ ]:
plt.plot(history.history['accuracy'],'r',label='training accuracy')
plt.plot(history.history['val_accuracy'],label='validation accuracy')
plt.xlabel('# epochs')
plt.ylabel('loss')
plt.legend()
plt.show()

In [21]:
#Prediction

In [3]:
from tensorflow.keras.models import load_model
import os
import cv2
import numpy as np

C:\Users\Siraj\anaconda3\Lib\site-packages\keras\src\export\tf2onnx_lib.py:8: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, "object"):


In [4]:
model = load_model('model-005.keras')

face_clsfr=cv2.CascadeClassifier(cv2.data.haarcascades + "haarcascade_frontalface_default.xml")

source=cv2.VideoCapture(0)

labels_dict={1:'With Mask',0:'Without Mask'}
color_dict={1:(0,255,0),0:(0,0,255)}

In [5]:
while(True):

    ret,img=source.read()
    gray=cv2.cvtColor(img,cv2.COLOR_BGR2GRAY)
    faces = face_clsfr.detectMultiScale(
        gray,
        scaleFactor=1.1,
        minNeighbors=5,
        
    )   

    for (x,y,w,h) in faces:
    
        face_img=gray[y:y+w,x:x+w]
        resized=cv2.resize(face_img,(100,100))
        normalized=resized/255.0
        reshaped=np.reshape(normalized,(1,100,100,1))
        result=model.predict(reshaped)

        label=np.argmax(result,axis=1)[0]
      
        cv2.rectangle(img,(x,y),(x+w,y+h),color_dict[label],2)
        cv2.rectangle(img,(x,y-50),(x+w,y),color_dict[label],-1)
        cv2.putText(img, labels_dict[label], (x, y-10),cv2.FONT_HERSHEY_SIMPLEX,0.8,(255,255,255),2)
        
        
    cv2.imshow('Mask Detection',img)
    key=cv2.waitKey(1)
    
    if(key==27):
        break
        
cv2.destroyAllWindows()
source.release()

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 354ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 68ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 57ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 51ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 56ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 74ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 52ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 57ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 53ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 71ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 60ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 62ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 51ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step
1/1 ━━━━━━━